In [1]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K

print("GPU available:", tf.config.list_physical_devices('GPU'))

2026-06-04 14:29:32.254035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780583372.435190      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780583372.487373      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780583372.922310      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780583372.922357      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780583372.922360      23 computation_placer.cc:177] computation placer alr

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# Thay đường dẫn này bằng đường dẫn Celeb-DF v2 trong thư mục /kaggle/input của bạn
DATASET_DIR = '/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2' 
IMG_SIZE = 224
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2 # Tách 20% dữ liệu để kiểm tra
)

train_gen = datagen.flow_from_directory(DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE), subset='training', class_mode='binary')
val_gen = datagen.flow_from_directory(DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE), subset='validation', class_mode='binary')

Found 80827 images belonging to 3 classes.
Found 20204 images belonging to 3 classes.


In [3]:
def build_cnn_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)
    return x

def build_mhsa_block(x):
    # Reshape để đưa vào Attention
    b, h, w, c = x.shape
    x_seq = layers.Reshape((h*w, c))(x)
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x_seq, x_seq)
    x = layers.Add()([x_seq, attn])
    x = layers.LayerNormalization()(x)
    return x

inputs = layers.Input(shape=(224, 224, 3))
x = build_cnn_block(inputs, 32)
x = build_cnn_block(x, 64)
x = build_cnn_block(x, 128)
x = build_mhsa_block(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

I0000 00:00:1780583524.337978      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780583524.344157      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [4]:
def focal_loss(alpha=0.25, gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return K.mean(-alpha * K.pow(1 - p_t, gamma) * K.log(p_t))
    return loss_fn

model.compile(optimizer='adam', loss=focal_loss(), metrics=['accuracy', tf.keras.metrics.AUC()])

In [5]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('/kaggle/working/best_model.keras', save_best_only=True, monitor='val_auc'),
    tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_auc')
]

history = model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=callbacks)

Epoch 1/20


I0000 00:00:1780583531.098242      82 service.cc:152] XLA service 0x7a2a7c001590 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780583531.098304      82 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780583531.098311      82 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780583531.938828      82 cuda_dnn.cc:529] Loaded cuDNN version 91002


   1/2526 ━━━━━━━━━━━━━━━━━━━━ 10:29:22 15s/step - accuracy: 0.5312 - auc: 0.4483 - loss: 0.0418

I0000 00:00:1780583541.540117      82 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1556s 610ms/step - accuracy: 0.7998 - auc: 0.5025 - loss: 0.0215 - val_accuracy: 0.8000 - val_auc: 0.5011 - val_loss: 0.0214
Epoch 2/20
2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1162s 460ms/step - accuracy: 0.8000 - auc: 0.4978 - loss: 0.0212 - val_accuracy: 0.8000 - val_auc: 0.4959 - val_loss: 0.0211
Epoch 3/20
2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1158s 458ms/step - accuracy: 0.8000 - auc: 0.4969 - loss: 0.0211 - val_accuracy: 0.8000 - val_auc: 0.4935 - val_loss: 0.0212
Epoch 4/20
2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1155s 457ms/step - accuracy: 0.8000 - auc: 0.4990 - loss: 0.0211 - val_accuracy: 0.8000 - val_auc: 0.5014 - val_loss: 0.0211
Epoch 5/20
2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1183s 468ms/step - accuracy: 0.8000 - auc: 0.5012 - loss: 0.0211 - val_accuracy: 0.8000 - val_auc: 0.5009 - val_loss: 0.0211
Epoch 6/20
2526/2526 ━━━━━━━━━━━━━━━━━━━━ 1310s 518ms/step - accuracy: 0.8000 - auc: 0.5009 - loss: 0.0212 - val_accuracy: 0.8000 - val_auc: 0.5020 - val_loss: 0.0211
Epoch 7/2